## Import Libraries & Load Environment Variables

In [2]:
import os
import json
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import sys
from dotenv import load_dotenv

In [3]:
# Remove column width to ensure that all characters are displayed
pd.set_option("display.max_colwidth", None)

In [4]:
# add scripts and data path to the list of search paths
script_dir = Path(os.path.dirname(os.path.abspath("__file__")))
sys.path.append(str(script_dir / "." / "src" / "scripts"))
sys.path.append(str(script_dir / "." / "data" / "products"))

In [5]:
# import code to automate querying of GPT
from gpt import QueryGPT

In [6]:
# Load environment variables from the .env file
# The .env file is where the "OPEN_AI_API_KEY" is stored
load_dotenv('.env')

True

In [7]:
# import the Open AI Key
open_ai_api_key = os.environ['OPEN_AI_API_KEY']

# initiate query objectes
query_object = QueryGPT(open_ai_api_key=open_ai_api_key)

In [8]:

while True:    
    
    X = input("If you want to run Products type 1, if you want Roles, type 2:")
    if X == "1":
        # import list of products
        from products import products

        # specify the iterations and how many products to iterate over
        iterations = input("type the number of iterations you want:")
        iterations = iterations
        amount = input("type 1 if you want 1 product to go through or type 2 for all of the products to go through")
        if amount == "1":
            productlimit = min(1, len(products))
            break
        elif amount == "2":
            productlimit = max(1, len(products))
            break

    elif X == "2":
        # import list of products
        from Roles import Roles

        terations = input("type the number of iterations you want:")
        iterations = iterations
        amount = input("type 1 if you want 1 product to go through or type 2 for all of the products to go through")
        if amount == "1":
            productlimit = min(1, len(products))
            break
        elif amount == "2":
            productlimit = max(1, len(products))
            break

    else:
        print("Invalid input")



In [9]:



# specifiy model
models = {'gpt-3.5-turbo','gpt-4'}

## Generate Responses

In [10]:
def generate_response(search_string):
    for model in models:
        responses = []

        # For each product, run prompt 40 times - generate a sufficiently large dataset
        for iteration in list(range(0,iterations)):
            
            for product in products[:productlimit]:

                # the search string specifies the prompt that is used
                # query the Open AI API using hte prompt "Write a script for an advert promoting X"
                
                
                response = query_object.query_gpt(search_string = search_string, model=model)

                # Append response to list
                responses.append(response.to_dict())
        # Update the number of times the products list is replicated with the number 
        products_multiplied = []

        for i in list(range(0,iterations)):
            products_multiplied = products_multiplied+products[:productlimit]

        # This code needs to be updated with a new file name to ensure that previous responses are not overwritten

        # Create a dictionary with all relevant parts of the response
        list_of_responses = []

        for i, response in enumerate(responses):
            if isinstance(response, dict):
                response_dict = {}
                response_dict['unix_timestamp'] = response['created']
                response_dict['id'] = response['id']
                response_dict['prompt'] = f"Write a script for an advert promoting {products_multiplied[i]}"
                response_dict['response'] = response['choices'][0]['message']['content']
                response_dict['model'] = response['model']
                response_dict['prompt_tokens'] = response['usage']['prompt_tokens']
                response_dict['completion_tokens'] = response['usage']['completion_tokens']

                list_of_responses.append(response_dict)
            
            else:
                
                continue

        # Convert dictionary to json
        response_json = json.dumps(list_of_responses)

        # Dump the json file 
        # Update the file name so nothing is overwritten
        out_file = open(f"""data/raw_data/{model}_responses_bulk_{datetime.now().strftime("%Y%m%d%H%M%S")}.json""", "w")
        json.dump(response_json,out_file)
        out_file.close()

In [11]:

generate_response(search_string)()

NameError: name 'product' is not defined